# 03 Train Module 3 ANN and local XGBoost

Run these notebooks in order, one cell at a time. Inspect each result before continuing.
Use a fresh Colab runtime for the pinned dependencies. CPU is sufficient for this small ANN;
GPU is optional. Use the same RUN_ID in all notebooks. Existing results are never overwritten.
The notebooks import repository code, so training and deployment use one implementation.
No model quality result is prefilled or claimed here.

In [ ]:
import sys, subprocess
from pathlib import Path
REPO = Path('/content/FedHealth-four-class')
BRANCH = 'feat/four-class-ml-federation'
if not REPO.exists():
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, '--single-branch',
                           'https://github.com/ahsen-dev86/FedHealth-FYP.git', str(REPO)])
else:
    branch = subprocess.check_output(['git', '-C', str(REPO), 'branch', '--show-current'], text=True).strip()
    if branch != BRANCH:
        raise RuntimeError(f'Use a fresh runtime: existing checkout is {branch}')
assert (3, 11) <= sys.version_info[:2] <= (3, 13), 'Use Python 3.11–3.13 (verified locally on 3.12)'
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO / 'Module 4 FL/requirements.txt')])
print('Dependencies installed. If Colab requests a restart, restart now, then run the next cell.')

In [ ]:
from pathlib import Path
import os, sys, json, subprocess
REPO = Path('/content/FedHealth-four-class')
from google.colab import drive
drive.mount('/content/drive')
# Keep this identical across all four notebooks. Use a new ID for a new experiment.
RUN_ID = 'four_class_v1'
RUN = Path('/content/drive/MyDrive/FYP/four_class') / RUN_ID
os.chdir(REPO)
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / 'Module 4 FL'))
from fedhealth_ml.data import Dataset
from fedhealth_ml.training import DEFAULT_CSV, DEFAULT_PROVENANCE
from fedhealth_ml.runtime import write_json
dataset = Dataset(DEFAULT_CSV, DEFAULT_PROVENANCE)
print('Run:', RUN)
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
print('Dataset:', dataset.sha256)

ANN: 10 → 64 → 32 → 4; dropout 0.30, L2 0.0001, Adam 0.001, batch 32, at most 100 epochs, validation-loss patience 10. XGBoost: at most 300 trees, depth 4, rate 0.05, validation patience 20. Temperature calibration uses validation only.

In [ ]:
from fedhealth_ml.training import train
CENTRAL = RUN / 'seed42' / 'centralized'
model_path = train(dataset, CENTRAL, seed=42, epochs=100, patience=10,
                   prepared=RUN / 'preprocessing')
print('Verified model:', model_path)

In [ ]:
import pandas as pd
report = json.loads((CENTRAL / 'validation_metrics.json').read_text())
display(pd.DataFrame({name: {k: m[k] for k in ['accuracy', 'macro_f1', 'log_loss', 'ece_15_bins']}
                      for name, m in report['metrics'].items()}).T)
for name, metrics in report['metrics'].items():
    print(name, 'recall:', metrics['per_class_recall'])
import matplotlib.pyplot as plt
history = json.loads((CENTRAL / 'history.json').read_text())
plt.plot(history['loss'], label='train'); plt.plot(history['val_loss'], label='validation')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.show()

Pause here and inspect the results. Test has not been evaluated. Share these validation metrics and curves before changing the protocol. Accuracy alone is insufficient; inspect leukemia recall, all four classes and the ANN versus hybrid comparison.